In [20]:
import os
os.environ['HSA_OVERRIDE_GFX_VERSION'] = '10.3.0'

import torch
print(torch.cuda.is_available())

False


# Input

In [21]:
raw_dummy = torch.tensor(
    [
        [
            -17.45557857, 2.174657688, 2.367182984, -16.99430584, -31.52132607,
        ],
        [
            -13.55773965, 1.965214473, -47.67187483, -5.831726978, -55.39959188,
        ],
        [
            -2.178412257, 9.753551178, -14.56978835, -76.58151401, -30.69740492,
        ],
        [
            -43.89739173, -50.37969167, -32.77935361, -66.49576026, -6.412497726,
        ],
        [
            -10.70635018, -72.22502322, -26.17916214, 1.4530419, 8.722773095,
        ],
        [
            -24.66762607, -18.16726717, 7.208516509, -13.64334883, -2.652998327,
        ],
    ]
)

In [22]:
raw_dummy_mean = torch.mean(raw_dummy, 0)
raw_dummy_std = torch.std(raw_dummy, 0, correction=0)

print(raw_dummy_mean)
print(raw_dummy_std)

tensor([-18.7439, -21.1464, -18.6041, -29.6823, -19.6602])
tensor([13.1362, 30.2911, 19.2597, 30.3075, 21.6416])


In [23]:
normal_dummy = (raw_dummy - raw_dummy_mean) / raw_dummy_std
print(normal_dummy)

tensor([[ 0.0981,  0.7699,  1.0889,  0.4186, -0.5481],
        [ 0.3948,  0.7630, -1.5093,  0.7870, -1.6514],
        [ 1.2611,  1.0201,  0.2095, -1.5474, -0.5100],
        [-1.9148, -0.9651, -0.7360, -1.2147,  0.6121],
        [ 0.6119, -1.6863, -0.3933,  1.0273,  1.3115],
        [-0.4509,  0.0984,  1.3402,  0.5292,  0.7859]])


In [24]:
def get_hook(name):
    """Returns a hook function that prints the gradient for the specified step."""
    def hook(grad):
        print(f"\n---> [GRAD DUMP] {name}")
        print(grad.detach().numpy())
    return hook

# Frontend

In [25]:
import torch
import torch.nn as nn


class FrontEnd(nn.Module):
    def __init__(self):
        super(FrontEnd, self).__init__()
        self.vert_conv = nn.Conv2d(
            in_channels=1, out_channels=1, kernel_size=(5, 3), padding="same"
        )
        self.vert_conv.weight = torch.nn.Parameter(
            torch.tensor(
                [
                    [-0.512, 0.897, -0.523],
                    [0.122, -0.204, -0.74],
                    [0.742, 0.805, -0.977],
                    [0.782, 0.516, -0.914],
                    [0.409, -0.438, 0.293],
                ]
            )
            .unsqueeze(0)
            .unsqueeze(0)
        )
        self.vert_conv.bias = torch.nn.Parameter(torch.zeros(self.vert_conv.bias.shape))
        self.vert_pool = nn.MaxPool2d(kernel_size=(1, 5))

        self.mean_pool_input = nn.AvgPool2d(kernel_size=(1, 5))
        self.horz_conv = nn.Conv2d(
            in_channels=1, out_channels=1, kernel_size=(3, 1), padding="same"
        )
        self.horz_conv.weight = torch.nn.Parameter(
            torch.tensor([[0.122], [0.516], [-0.977]]).unsqueeze(0).unsqueeze(0)
        )
        self.horz_conv.bias = torch.nn.Parameter(torch.zeros(self.horz_conv.bias.shape))
        self.relu = nn.ReLU()

    def forward(self, x):
        x.register_hook(get_hook("FrontEnd: Input (x)"))

        # Vertical conv
        v = self.vert_conv(x)
        v.register_hook(get_hook("FrontEnd: vert_conv pre-ReLU"))

        v_relu = self.relu(v)
        v_relu.register_hook(get_hook("FrontEnd: vert_conv post-ReLU"))

        v_pool = self.vert_pool(v_relu)
        v_pool.register_hook(get_hook("FrontEnd: vert_pool out"))

        # Horizontal conv
        h = self.mean_pool_input(x)
        h.register_hook(get_hook("FrontEnd: horz_mean_pool input"))

        h_conv = self.horz_conv(h)
        h_conv.register_hook(get_hook("FrontEnd: horz_conv pre-ReLU"))

        h_relu = self.relu(h_conv)
        h_relu.register_hook(get_hook("FrontEnd: horz_conv post-ReLU"))

        out = torch.cat([v_pool, h_relu], dim=-1)
        out.register_hook(get_hook("FrontEnd: Output (Concatenated)"))
        return out

# CNN backend

In [26]:
class CNNBackEnd(nn.Module):
    def __init__(self):
        super(CNNBackEnd, self).__init__()
        self.conv = nn.Conv2d(
            in_channels=1, out_channels=1, kernel_size=(3, 2), padding="same"
        )
        self.conv.weight = torch.nn.Parameter(
            torch.tensor([[-0.204, -0.74], [0.805, -0.977], [0.516, -0.914]])
            .unsqueeze(0)
            .unsqueeze(0)
        )
        self.conv.bias = torch.nn.Parameter(torch.zeros(self.conv.bias.shape))
        self.relu = nn.ReLU()

    def forward(self, x):
        x.register_hook(get_hook("CNNBackEnd: Input"))

        x_conv = self.conv(x)
        x_conv.register_hook(get_hook("CNNBackEnd: conv pre-ReLU"))

        x_relu = self.relu(x_conv)
        x_relu.register_hook(get_hook("CNNBackEnd: conv post-ReLU"))

        # Note: Added squeeze(1) before permute to avoid PyTorch dimensions crash
        x_sq = x_relu.squeeze(1)
        x_sq.register_hook(get_hook("CNNBackEnd: squeezed output"))

        x_perm = torch.permute(x_sq, (0, 2, 1))
        x_perm.register_hook(get_hook("CNNBackEnd: permute"))

        p_max, _ = torch.max(x_perm, dim=2)
        p_max.register_hook(get_hook("CNNBackEnd: max_pool"))

        p_mean = torch.mean(x_perm, dim=2)
        p_mean.register_hook(get_hook("CNNBackEnd: mean_pool"))

        out = torch.cat([p_max, p_mean], dim=1)
        out.register_hook(get_hook("CNNBackEnd: Output (Concat)"))
        return out

# GRU backend

In [27]:
class GRUBackEnd(nn.Module):
    def __init__(self):
        super(GRUBackEnd, self).__init__()
        self.gru = nn.GRU(input_size=2, hidden_size=2, num_layers=1, batch_first=True)
        custom_weight_ih = torch.tensor(
            [
                [0.878, -0.139],
                [-0.013, 0.997],
                [-0.212, 0.248],
                [-0.253, -0.872],
                [-0.507, 0.891],
                [0.672, 0.741],
            ],
            dtype=torch.float32,
        )
        custom_weight_hh = torch.tensor(
            [
                [-0.253, -0.872],
                [-0.212, 0.248],
                [-0.013, 0.997],
                [0.878, -0.139],
                [0.672, 0.741],
                [-0.507, 0.891],
            ],
            dtype=torch.float32,
        )
        self.gru.weight_ih_l0 = torch.nn.Parameter(custom_weight_ih)
        self.gru.weight_hh_l0 = torch.nn.Parameter(custom_weight_hh)
        self.gru.bias_ih_l0 = torch.nn.Parameter(torch.zeros(3 * 2))
        self.gru.bias_hh_l0 = torch.nn.Parameter(torch.zeros(3 * 2))

    def forward(self, x):
        x.register_hook(get_hook("GRUBackEnd: Input"))

        x_sq = x.squeeze(1)
        x_sq.register_hook(get_hook("GRUBackEnd: squeeze"))

        out, _ = self.gru(x_sq, torch.tensor([[0.805, -0.977]]).unsqueeze(0))
        out.register_hook(get_hook("GRUBackEnd: GRU out"))

        p_max, _ = torch.max(out, dim=1)
        p_max.register_hook(get_hook("GRUBackEnd: max_pool"))

        p_mean = torch.mean(out, dim=1)
        p_mean.register_hook(get_hook("GRUBackEnd: mean_pool"))

        out_cat = torch.cat([p_max, p_mean], dim=1)
        out_cat.register_hook(get_hook("GRUBackEnd: Output (Concat)"))
        return out_cat

# Attention backend

In [28]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model=2, max_len=6):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        N = 10000.0
        i = torch.arange(0, d_model // 2)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.pow(N, (2 * i) / d_model)
        pe[:, 0::2] = torch.sin(position / div_term)
        if d_model > 1:
            pe[:, 1::2] = torch.cos(position / div_term)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, : x.size(1), :].to(x.device)


class AttentionBackEnd(nn.Module):
    def __init__(self):
        super(AttentionBackEnd, self).__init__()
        self.pos_encoder = PositionalEncoding(d_model=2)
        self.attention = nn.MultiheadAttention(
            embed_dim=2, num_heads=2, batch_first=True
        )
        self.ffn = nn.Sequential(nn.Linear(2, 2), nn.ReLU(), nn.Linear(2, 2))

        wq = torch.tensor([[-0.862, 0.880], [-0.487, 0.238]], dtype=torch.float32)
        wk = torch.tensor([[0.409, -0.728], [0.251, -0.082]], dtype=torch.float32)
        wv = torch.tensor([[-0.052, -0.648], [0.945, 0.435]], dtype=torch.float32)
        self.attention.in_proj_weight = torch.nn.Parameter(
            torch.cat([wq, wk, wv], dim=0)
        )
        self.attention.in_proj_bias = torch.nn.Parameter(torch.zeros(6))
        w_out = torch.tensor([[-0.474, -0.068], [0.534, 0.788]], dtype=torch.float32).T
        self.attention.out_proj.weight = torch.nn.Parameter(w_out)
        self.attention.out_proj.bias = torch.nn.Parameter(torch.zeros(2))
        self.ffn[0]._parameters["weight"] = torch.nn.Parameter(
            torch.tensor([[-0.474, 0.251], [0.945, 0.238]], dtype=torch.float32).T
        )
        self.ffn[0]._parameters["bias"] = torch.nn.Parameter(torch.zeros(2))
        self.ffn[2]._parameters["weight"] = torch.nn.Parameter(
            torch.tensor([[-0.728, -0.862], [-0.648, 0.052]], dtype=torch.float32).T
        )
        self.ffn[2]._parameters["bias"] = torch.nn.Parameter(torch.zeros(2))

    def forward(self, x):
        x.register_hook(get_hook("AttentionBackEnd: Input"))

        x_sq = x.squeeze(1)
        x_sq.register_hook(get_hook("AttentionBackEnd: squeeze"))

        x_pe = self.pos_encoder(x_sq)
        x_pe.register_hook(get_hook("AttentionBackEnd: PE out"))

        attn_out, _ = self.attention(x_pe, x_pe, x_pe)
        attn_out.register_hook(get_hook("AttentionBackEnd: attn_out"))

        res_attn = x_pe + attn_out
        res_attn.register_hook(get_hook("AttentionBackEnd: res_attn"))

        ffn_out = self.ffn(res_attn)
        ffn_out.register_hook(get_hook("AttentionBackEnd: ffn_out"))

        res_ffn = res_attn + ffn_out
        res_ffn.register_hook(get_hook("AttentionBackEnd: res_ffn"))

        p_max, _ = torch.max(res_ffn, dim=1)
        p_max.register_hook(get_hook("AttentionBackEnd: max_pool"))

        p_mean = torch.mean(res_ffn, dim=1)
        p_mean.register_hook(get_hook("AttentionBackEnd: mean_pool"))

        out = torch.cat([p_max, p_mean], dim=1)
        out.register_hook(get_hook("AttentionBackEnd: Output (Concat)"))
        return out

# Classifier

In [29]:
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        self.fc = nn.Linear(in_features=12, out_features=3)
        self.fc.weight = torch.nn.Parameter(
            torch.tensor(
                [
                    [-0.134, -0.671, -0.778],
                    [-0.945, -0.04, 0.991],
                    [-0.62, 0.312, -0.121],
                    [-0.291, -0.202, 0.186],
                ]
            ).T
        )
        self.fc.bias = torch.nn.Parameter(torch.zeros(self.fc.bias.shape))
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x.register_hook(get_hook("Classifier: Input (From Backend Concat)"))

        z = self.fc(x)
        z.register_hook(get_hook("Classifier: pre-sigmoid (z_O)"))

        y = self.sigmoid(z)
        y.register_hook(get_hook("Classifier: post-sigmoid (y_n / pred)"))
        return y


class MusicModel(nn.Module):
    def __init__(self, backend_type="cnn"):
        super(MusicModel, self).__init__()
        self.frontend = FrontEnd()
        if backend_type == "cnn":
            self.backend = CNNBackEnd()
        elif backend_type == "gru":
            self.backend = GRUBackEnd()
        elif backend_type == "attention":
            self.backend = AttentionBackEnd()
        self.classifier = Classifier()

    def forward(self, x):
        x = self.frontend(x)
        x = self.backend(x)
        out = self.classifier(x)
        return out


x = normal_dummy.detach().clone()
x.requires_grad_(True)
x = x.unsqueeze(0)

model_cnn = MusicModel(backend_type="cnn")
model_gru = MusicModel(backend_type="gru")
model_attn = MusicModel(backend_type="attention")

pred_cnn = model_cnn(x)
pred_gru = model_gru(x)
pred_attn = model_attn(x)

print("CNN+CNN predictions (gembira, sedih, tegang):", pred_cnn.detach().numpy())
print("CNN+GRU predictions (gembira, sedih, tegang):", pred_gru.detach().numpy())
print("CNN+Attn predictions (gembira, sedih, tegang):", pred_attn.detach().numpy())

CNN+CNN predictions (gembira, sedih, tegang): [[0.09093296 0.12871003 0.09472395]]
CNN+GRU predictions (gembira, sedih, tegang): [[0.33157495 0.48198342 0.82506484]]
CNN+Attn predictions (gembira, sedih, tegang): [[0.00817633 0.11878848 0.41308215]]


# Backpropagation

In [35]:
# Dummy target label (gembira, sedih, tegang)
y_true = torch.tensor([[0.0, 1.0, 1.0]], dtype=torch.float32)

# Dummy frequencies of the positive class in the training set (p_i)
p_frequencies = torch.tensor(
    [0.115384615, 0.615384615, 0.269230769], dtype=torch.float32
)


def bour_weighted_bce_loss(predictions, targets, p):
    C = predictions.size(-1)
    weight_pos = 2.0 / (1.0 + p)
    weight_neg = (2.0 * p) / (1.0 + p)
    eps = 10e-12
    preds_clamped = torch.clamp(predictions, eps, 1.0 - eps)
    term1 = weight_pos * targets * torch.log(preds_clamped)
    term2 = weight_neg * (1.0 - targets) * torch.log(1.0 - preds_clamped)
    loss = -torch.sum(term1 + term2) / C
    return loss


def perform_manual_update_and_print(model_name, model, predictions, targets, p, lr=0.1):
    model.zero_grad()
    loss = bour_weighted_bce_loss(predictions, targets, p)

    print("\n" + "=" * 101)
    print(f"BACKPROPAGATION DUMP FOR: {model_name}")
    print(f"Loss: {loss.item()}")
    print("=" * 101)

    loss.backward(retain_graph=True)

    print("\n--- Parameter Updates ---")
    for name, param in model.named_parameters():
        if param.requires_grad and param.grad is not None:
            updated_value = param.data - lr * param.grad
            print(f"[{name}] Gradient:\n{param.grad.detach().numpy()}")
            print(f"[{name}] Updated Value:\n{updated_value.detach().numpy()}\n")


learning_rate = 0.1

perform_manual_update_and_print(
    "CNN+CNN Model", model_cnn, pred_cnn, y_true, p_frequencies, lr=learning_rate
)
perform_manual_update_and_print(
    "CNN+GRU Model", model_gru, pred_gru, y_true, p_frequencies, lr=learning_rate
)
perform_manual_update_and_print(
    "CNN+Attention Model",
    model_attn,
    pred_attn,
    y_true,
    p_frequencies,
    lr=learning_rate,
)


BACKPROPAGATION DUMP FOR: CNN+CNN Model
Loss: 2.0905954837799072

---> [GRAD DUMP] Classifier: post-sigmoid (y_n / pred)
[[ 0.07586407 -3.2064202  -5.5450864 ]]

---> [GRAD DUMP] Classifier: pre-sigmoid (z_O)
[[ 0.00627124 -0.35958    -0.47549853]]

---> [GRAD DUMP] CNNBackEnd: Output (Concat)
[[ 0.6103757  -0.46276215 -0.05854181 -0.01763249]]

---> [GRAD DUMP] Classifier: Input (From Backend Concat)
[[ 0.6103757  -0.46276215 -0.05854181 -0.01763249]]

---> [GRAD DUMP] CNNBackEnd: mean_pool
[[-0.05854181 -0.01763249]]

---> [GRAD DUMP] CNNBackEnd: max_pool
[[ 0.6103757  -0.46276215]]

---> [GRAD DUMP] CNNBackEnd: permute
[[[-0.00975697 -0.00975697 -0.00975697 -0.00975697  0.6006187
   -0.00975697]
  [-0.00293875 -0.00293875 -0.4657009  -0.00293875 -0.00293875
   -0.00293875]]]

---> [GRAD DUMP] CNNBackEnd: squeezed output
[[[-0.00975697 -0.00293875]
  [-0.00975697 -0.00293875]
  [-0.00975697 -0.4657009 ]
  [-0.00975697 -0.00293875]
  [ 0.6006187  -0.00293875]
  [-0.00975697 -0.002938

Scratch below please ignore

In [31]:
test1 = torch.tensor([[1.,2.,3.],[2.,7.,8.],[3.,6.,5.]])
test2 = torch.tensor([[5.,6.,3.],[4.,5.,3.],[7.,7.,8.]])

test3 = -torch.sum(test1+test2)/(3*3)
print(torch.isnan(test3))
if test3.isnan():
    print("Hello true")
if not test3.isnan():
    print("Hello false")

print(-torch.mean(test1+test2))
print(-torch.sum(test1+test2)/(3*3))
print(-torch.sum(test1+test2)/(3*2))
print(-torch.sum(test2, dim=1)/(3))
print(-torch.mean(test1+test2) * 0 + 1e-3)
print(torch.tensor(10e-12).item())
print(((test1+test2)*0.0).sum())

tensor(False)
Hello false
tensor(-9.4444)
tensor(-9.4444)
tensor(-14.1667)
tensor([-4.6667, -4.0000, -7.3333])
tensor(0.0010)
9.999999960041972e-12
tensor(0.)
